# Woche 6 – Freitag: Abschluss Integrationstheorie & VaR – Ausblick auf ES

## Block 1 (06:00 – 08:45): Wiederholung und Ausblick auf Expected Shortfall

**Aufgabe:** Berechnen Sie den historischen VaR (99%) und den Expected Shortfall (ES) für die `amount`‑Spalte. ES ist der Durchschnitt der Verluste, die den VaR überschreiten.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_json('cleaned_aml_batch.json')
losses = df['amount'].values
alpha = 0.99
var = np.percentile(losses, alpha * 100)
es = losses[losses >= var].mean()
print(f'VaR (99%): {var:.2f}, ES: {es:.2f}')


> **Verbesserungshinweis:** ES ist kohärent (subadditiv) und wird in Basel III/IV gefordert. In Woche 9 vertiefen wir Risikomaße.

---

## Block 2 (09:30 – 11:40): Vergleich Autoencoder vs. MLP für Anomalieerkennung

**Aufgabe:** Trainieren Sie sowohl den linearen Autoencoder als auch das MLP (beide mit gleicher Bottleneck‑Dimension). Vergleichen Sie die Rekonstruktionsfehler für die höchste Transaktion. Welches Modell erkennt die Ausreißer besser?

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_json('cleaned_aml_batch.json')
features = df[['amount']].copy()
features['amount_log'] = np.log1p(features['amount'])
scaler = StandardScaler()
X = torch.tensor(scaler.fit_transform(features), dtype=torch.float32)

# Linearer Autoencoder
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(2, 1)
        self.decoder = nn.Linear(1, 2)
    def forward(self, x):
        return self.decoder(self.encoder(x))

# MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 8)
        self.fc2 = nn.Linear(8, 4)
        self.fc3 = nn.Linear(4, 2)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

def train_and_get_errors(model_class, X, epochs=200):
    model = model_class()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    for epoch in range(epochs):
        out = model(X)
        loss = criterion(out, X)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        rec = model(X)
        errors = torch.mean((X - rec)**2, dim=1).numpy()
    return errors

ae_errors = train_and_get_errors(AE, X)
mlp_errors = train_and_get_errors(MLP, X)
print(f'Max AE error: {ae_errors.max():.4f}')
print(f'Max MLP error: {mlp_errors.max():.4f}')


> **Verbesserungshinweis:** MLPs sind flexibler, können aber auch überanpassen (overfitting) – bei kleinen Datensätzen oft linearer AE besser.

---

## Block 3 (12:40 – 14:10): GOV – Zusammenfassung Art. 16‑21

Erstellen Sie eine einseitige Zusammenfassung der Pflichten aus Art. 16‑21 für ein fiktives KI‑Startup, das einen AML‑Autoencoder entwickelt. Speichern Sie als `27_zusammenfassung_art16_21.md` in Track_C.

---

## Block 4 (14:40 – 16:00): Passiv – Abschlusstest (Buscha Kap. 1.5.1)

Bearbeiten Sie den Abschlusstest zum Passiv. Schreiben Sie einen kurzen Aufsatz (1 Seite) über die Vor- und Nachteile des VaR im Vergleich zum ES – ausschließlich im Passiv. Speichern Sie als `28_passiv_aufsatz_var_es.md` in Track_C.